In [2]:
import pandas as pd
import numpy as np
import getpass
from sqlalchemy import create_engine, URL

url = URL.create(
    "postgresql+psycopg2",
    username="postgres",
    password=getpass.getpass("PostgreSQL şifresi: "),
    host="localhost", port=5432, database="retail_scorecard",
)
engine = create_engine(url)

df = pd.read_sql("SELECT * FROM clean.lc_model WHERE sample IN ('train', 'validation')", engine)
df.groupby("sample")["bad"].agg(["count", "sum", "mean"])


,count,sum,mean
sample,,,
train,120683,15007,0.124351
validation,162570,22315,0.137264


In [3]:
from optbinning import BinningProcess

pd.set_option("display.max_rows", 100)

exclude = ["id", "issue_date", "sample", "bad", "bad_alt", "lc_grade"]
categorical = ["purpose", "home_ownership", "verification_status"]
features = [c for c in df.columns if c not in exclude]
numeric = [c for c in features if c not in categorical]

df[numeric] = df[numeric].astype(float)

train = df[df["sample"] == "train"]
valid = df[df["sample"] == "validation"]

bp = BinningProcess(
    variable_names=features,
    categorical_variables=categorical,
    binning_fit_params={v: {"monotonic_trend": "auto_asc_desc"} for v in numeric},
)
bp.fit(train[features], train["bad"])

iv = bp.summary().sort_values("iv", ascending=False)
iv[["name", "n_bins", "iv", "gini"]]

,name,n_bins,iv,gini
9,fico,13,0.129429,0.187563
4,annual_inc,13,0.103354,0.182442
29,bc_open_to_buy,9,0.084357,0.151567
56,tot_hi_cred_lim,10,0.080994,0.155177
28,avg_cur_bal,12,0.0738,0.143925
58,total_bc_limit,13,0.069837,0.138631
55,tot_cur_bal,11,0.065922,0.134994
60,total_rev_hi_lim,10,0.057057,0.122914
6,dti,11,0.048483,0.125818
7,loan_to_income,9,0.047358,0.121636


In [4]:
cand = iv.loc[iv["iv"] >= 0.02, "name"].tolist()

woe_train = bp.transform(train[features], metric="woe")[cand]
corr = woe_train.corr()

upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
pairs = upper.stack().reset_index()
pairs.columns = ["var1", "var2", "corr"]
pairs[pairs["corr"].abs() > 0.6].sort_values("corr", key=abs, ascending=False)


c:\Users\demir\retail-scorecard\.venv\lib\site-packages\optbinning\binning\transformations.py:38: RuntimeWarning: invalid value encountered in log
  return np.log((1. / event_rate - 1) * n_event / n_nonevent)


,var1,var2,corr
375,num_rev_tl_bal_gt_0,num_actv_rev_tl,0.997007
80,tot_hi_cred_lim,tot_cur_bal,0.960970
103,avg_cur_bal,tot_cur_bal,0.898783
349,mo_sin_old_rev_tl_op,credit_hist_months,0.877013
126,total_bc_limit,total_rev_hi_lim,0.871226
78,tot_hi_cred_lim,avg_cur_bal,0.865319
55,bc_open_to_buy,total_bc_limit,0.785673
312,percent_bc_gt_75,bc_util,0.761599
351,mths_since_recent_inq,inq_last_6mths,0.751947
326,bc_util,revol_util,0.744992


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

selected = ["fico", "annual_inc", "tot_hi_cred_lim", "bc_open_to_buy", "dti",
            "loan_to_income", "mort_acc", "acc_open_past_24mths",
            "mths_since_recent_inq", "percent_bc_gt_75", "mo_sin_old_rev_tl_op", "purpose"]

# empty bins (e.g. no missing values in train) get a neutral WoE of 0
X_train = bp.transform(train[features], metric="woe")[selected].fillna(0)
X_valid = bp.transform(valid[features], metric="woe")[selected].fillna(0)

lr = LogisticRegression(penalty=None, max_iter=1000)
lr.fit(X_train, train["bad"])

gini_train = 2 * roc_auc_score(train["bad"], lr.predict_proba(X_train)[:, 1]) - 1
gini_valid = 2 * roc_auc_score(valid["bad"], lr.predict_proba(X_valid)[:, 1]) - 1
print(f"Gini train: {gini_train:.3f}   Gini validation: {gini_valid:.3f}")

pd.Series(lr.coef_[0], index=selected).sort_values()

c:\Users\demir\retail-scorecard\.venv\lib\site-packages\optbinning\binning\transformations.py:38: RuntimeWarning: invalid value encountered in log
  return np.log((1. / event_rate - 1) * n_event / n_nonevent)
c:\Users\demir\retail-scorecard\.venv\lib\site-packages\optbinning\binning\transformations.py:38: RuntimeWarning: invalid value encountered in log
  return np.log((1. / event_rate - 1) * n_event / n_nonevent)


Gini train: 0.328   Gini validation: 0.323


acc_open_past_24mths    -1.060325
mths_since_recent_inq   -1.010672
purpose                 -0.954690
loan_to_income          -0.861701
fico                    -0.563720
dti                     -0.480454
tot_hi_cred_lim         -0.466106
bc_open_to_buy          -0.445285
annual_inc              -0.364872
mo_sin_old_rev_tl_op    -0.224437
percent_bc_gt_75        -0.203591
mort_acc                -0.129836
dtype: float64

In [6]:
print(f"Gini train: {gini_train:.3f}   Gini validation: {gini_valid:.3f}")


Gini train: 0.328   Gini validation: 0.323


In [7]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

Xc = sm.add_constant(X_train)
logit = sm.Logit(train["bad"], Xc).fit(disp=0)

vif = pd.Series(
    [variance_inflation_factor(Xc.values, i) for i in range(1, Xc.shape[1])],
    index=selected,
)

pd.DataFrame({
    "coef": logit.params[selected].round(3),
    "p_value": logit.pvalues[selected].round(4),
    "vif": vif.round(2),
}).sort_values("p_value")


,coef,p_value,vif
fico,-0.564,0.0000,1.60
annual_inc,-0.365,0.0000,1.87
tot_hi_cred_lim,-0.474,0.0000,2.50
bc_open_to_buy,-0.429,0.0000,1.94
dti,-0.488,0.0000,1.26
loan_to_income,-0.852,0.0000,1.26
acc_open_past_24mths,-1.059,0.0000,1.15
mths_since_recent_inq,-1.013,0.0000,1.03
purpose,-0.957,0.0000,1.02
mo_sin_old_rev_tl_op,-0.238,0.0000,1.25


In [8]:
sel_11 = [v for v in selected if v != "mort_acc"]

lr11 = LogisticRegression(penalty=None, max_iter=1000).fit(X_train[sel_11], train["bad"])
g_tr = 2 * roc_auc_score(train["bad"], lr11.predict_proba(X_train[sel_11])[:, 1]) - 1
g_va = 2 * roc_auc_score(valid["bad"], lr11.predict_proba(X_valid[sel_11])[:, 1]) - 1
print(f"11 variables -> Gini train: {g_tr:.3f}   validation: {g_va:.3f}")

11 variables -> Gini train: 0.328   validation: 0.323
